# ARIZA Aggregate Analiz

**Kaynak:** `panel_data/iett_data.db` → `ariza` tablosu (78.300 kayıt, 2025 Oca-Haz, TÜM operatörler: İETT + ÖHO + KOOP)

**Çıktı:** `panel_data/ariza_ozet.json` — Flask `/api/panel/ariza_ozet` endpoint'i bu dosyayı okur.

**Ciddi Arıza Tanımı:** `SONUCTIPI IN ('Yol Ustası - Garaj', 'Çekilerek - Garaj', 'Telefonla - Garaj', 'Kayıtsız - Garaj', 'Oto Değişimi')` — yani araç garaja gitmek zorunda kaldı veya değiştirildi (operasyonel etki).

In [10]:
import sqlite3, json, os
from pathlib import Path

ROOT = Path.cwd()
# Notebook ARIZA/ altinda ise root iki yukari
if ROOT.name == 'ARIZA':
    ROOT = ROOT.parent
DB_PATH = ROOT / 'panel_data' / 'iett_data.db'
OUT_PATH = ROOT / 'panel_data' / 'ariza_ozet.json'
print('DB:', DB_PATH, '\nOUT:', OUT_PATH)

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

# Ciddi arıza tanımı — operasyonel etki bazlı
# DB'deki gerçek SONUCTIPI değerleri: "Yol Ustası - Garaj", "Çekilerek - Garaj",
# "Telefonla - Garaj", "Kayıtçı - Garaj" (NOT: önceki versiyon yanlış olarak "Kayıtsız" diyordu,
# 2,438 kayıt eksik sayıldı), "Oto Değişimi"
CIDDI_SONUCLAR = ('Yol Ustası - Garaj', 'Çekilerek - Garaj', 'Telefonla - Garaj',
                  'Kayıtçı - Garaj', 'Oto Değişimi')

DB: c:\Users\asus\Desktop\Datathon\panel_data\iett_data.db 
OUT: c:\Users\asus\Desktop\Datathon\panel_data\ariza_ozet.json


## 1) KPI Hesabı

In [11]:
cur.execute('SELECT COUNT(*) FROM ariza')
toplam = cur.fetchone()[0]

cur.execute(f"SELECT COUNT(*) FROM ariza WHERE SONUCTIPI IN {CIDDI_SONUCLAR}")
ciddi_n = cur.fetchone()[0]

cur.execute('SELECT SUM(ZAYISEFERSAYISI) FROM ariza WHERE ZAYISEFERSAYISI>0')
zayi_top = cur.fetchone()[0] or 0

# Medyan tepki süresi (tepki süresi sıralanır, ortadaki alınır)
cur.execute('SELECT TEPKI_SURE_DK FROM ariza WHERE TEPKI_SURE_DK IS NOT NULL AND TEPKI_SURE_DK>0 AND TEPKI_SURE_DK<1440 ORDER BY TEPKI_SURE_DK')
tepki = [r[0] for r in cur.fetchall()]
medyan_tepki = tepki[len(tepki)//2] if tepki else 0

kpi = {
    'toplam_ariza': toplam,
    'ciddi_ariza': ciddi_n,
    'ciddi_ariza_pct': round(ciddi_n/toplam*100, 1),
    'medyan_tepki_dk': round(medyan_tepki, 0),
    'toplam_zayi_sefer': zayi_top,
}
kpi

{'toplam_ariza': 78300,
 'ciddi_ariza': 26367,
 'ciddi_ariza_pct': 33.7,
 'medyan_tepki_dk': 41.0,
 'toplam_zayi_sefer': 45190}

## 2) Tip Bazlı Dağılım (Top 20) — ARIZAUSTKODTANIM

In [12]:
cur.execute(f'''SELECT ARIZAUSTKODTANIM, COUNT(*) toplam,
                SUM(CASE WHEN SONUCTIPI IN {CIDDI_SONUCLAR} THEN 1 ELSE 0 END) ciddi
                FROM ariza
                WHERE ARIZAUSTKODTANIM IS NOT NULL
                  AND ARIZAUSTKODTANIM NOT IN ('Sınıflandırılmadı','Sınıflandırıldı','Destek','Belirsiz')
                GROUP BY ARIZAUSTKODTANIM ORDER BY toplam DESC LIMIT 20''')
tip_bazli = [{'tip': r[0], 'sayi': r[1], 'ciddi': r[2],
              'ciddi_pct': round(r[2]/r[1]*100, 1) if r[1] else 0}
             for r in cur.fetchall()]
tip_bazli[:5]

[{'tip': 'SOĞUTMA SİSTEMİ ARIZASI',
  'sayi': 6601,
  'ciddi': 1995,
  'ciddi_pct': 30.2},
 {'tip': 'KAPI ARIZALARI', 'sayi': 6334, 'ciddi': 2112, 'ciddi_pct': 33.3},
 {'tip': 'ELEKTRİK SİSTEMİ ARIZALARI',
  'sayi': 6160,
  'ciddi': 1780,
  'ciddi_pct': 28.9},
 {'tip': 'MOTOR ARIZALARI', 'sayi': 5822, 'ciddi': 3305, 'ciddi_pct': 56.8},
 {'tip': 'KAROSER ARIZALARI', 'sayi': 4340, 'ciddi': 1547, 'ciddi_pct': 35.6}]

## 3) Saatlik Dağılım — SAAT_BANDI

In [13]:
# SAAT_BANDI: ORDER BY ile alfabetik sirayla gelir (Aksam, Gece, Gece geç, Sabah, Öğlen) — anlamsız.
# CASE WHEN ile kronolojik sira (Gece → Sabah → Öğlen → Aksam → Gece geç)
cur.execute("""
    SELECT SAAT_BANDI, COUNT(*) FROM ariza
    WHERE SAAT_BANDI IS NOT NULL
    GROUP BY SAAT_BANDI
    ORDER BY CASE SAAT_BANDI
        WHEN 'Gece (00-05)'     THEN 1
        WHEN 'Sabah (06-11)'    THEN 2
        WHEN 'Öğlen (12-17)'    THEN 3
        WHEN 'Akşam (18-20)'    THEN 4
        WHEN 'Gece geç (21-23)' THEN 5
        ELSE 99
    END
""")
saat_bazli = [{'saat_bandi': r[0], 'sayi': r[1]} for r in cur.fetchall()]
saat_bazli

[{'saat_bandi': 'Gece (00-05)', 'sayi': 2006},
 {'saat_bandi': 'Sabah (06-11)', 'sayi': 32252},
 {'saat_bandi': 'Öğlen (12-17)', 'sayi': 30341},
 {'saat_bandi': 'Akşam (18-20)', 'sayi': 9829},
 {'saat_bandi': 'Gece geç (21-23)', 'sayi': 3872}]

## 4) Aylık Trend

In [14]:
cur.execute('SELECT AY,COUNT(*) FROM ariza GROUP BY AY ORDER BY AY')
aylik = [{'ay': r[0], 'sayi': r[1]} for r in cur.fetchall()]
aylik

[{'ay': 1, 'sayi': 11942},
 {'ay': 2, 'sayi': 12953},
 {'ay': 3, 'sayi': 12789},
 {'ay': 4, 'sayi': 11555},
 {'ay': 5, 'sayi': 14525},
 {'ay': 6, 'sayi': 14536}]

## 5) Garaj Bazlı Ciddi Arıza % (≥50 arıza kayıtlı garajlar)

In [15]:
# GARAJ × ARAÇ × CİDDİ + MARKA DAĞILIMI (zenginleştirilmiş)
# Eskiden sadece ciddi% gösteriyordu — büyük/küçük garaj karşılaştırması adil değildi
# (ADALAR 412 ariza ile %73.3 ciddi → 1. sıraya geçiyordu ama mutlak küçüktü).
# Yeni: araç başına ciddi (ciddi/arac) = adil oran. + her garajın marka dağılımı (stacked bar için).

# 1) Garaj başına KPI'lar
cur.execute(f'''
    SELECT GARAJADI,
           COUNT(DISTINCT KAPINO) AS arac,
           COUNT(*) AS ariza,
           SUM(CASE WHEN SONUCTIPI IN {CIDDI_SONUCLAR} THEN 1 ELSE 0 END) AS ciddi
    FROM ariza
    WHERE GARAJADI IS NOT NULL
      AND GARAJADI != ''
      AND GARAJADI != 'Kayıtsız Araç'
      AND KAPINO IS NOT NULL
    GROUP BY GARAJADI
    HAVING arac >= 10
    ORDER BY (CAST(ciddi AS REAL) / arac) DESC
''')
garaj_kpi = cur.fetchall()
top_garaj_isimleri = [r[0] for r in garaj_kpi]

# 2) Her garaj icin MARKA dagilimi (MODEL'in ilk kelimesinden imputed)
ph_g = ','.join(['?'] * len(top_garaj_isimleri))
cur.execute(f'''
    SELECT GARAJADI,
        CASE
            WHEN MARKA='Kayıtsız Araç' AND (MODEL LIKE 'BMC%' OR MODEL='BMC')                          THEN 'BMC'
            WHEN MARKA='Kayıtsız Araç' AND (MODEL LIKE 'OTOKAR%' OR MODEL='OTOKAR')                    THEN 'OTOKAR'
            WHEN MARKA='Kayıtsız Araç' AND (MODEL LIKE 'MERCEDES%' OR MODEL='MERCEDES' OR MODEL LIKE 'CİTİPORT%') THEN 'MERCEDES'
            WHEN MARKA='Kayıtsız Araç' AND (MODEL LIKE 'TEMSA%' OR MODEL='TEMSA')                      THEN 'TEMSA'
            WHEN MARKA='Kayıtsız Araç' AND (MODEL LIKE 'GÜLERYÜZ%' OR MODEL LIKE 'COBRA%')             THEN 'GÜLERYÜZ'
            WHEN MARKA='Kayıtsız Araç' AND (MODEL LIKE 'KARSAN%' OR MODEL='KARSAN')                    THEN 'KARSAN'
            WHEN MARKA='Kayıtsız Araç' AND (MODEL LIKE 'ANADOLU%' OR MODEL LIKE 'ISUZU%')              THEN 'ISUZU'
            WHEN MARKA='Kayıtsız Araç' AND (MODEL LIKE 'AKIA%' OR MODEL='AKIA')                        THEN 'AKIA'
            WHEN MARKA='Kayıtsız Araç' AND MODEL LIKE 'TEZELLER%'                                       THEN 'TEZELLER'
            WHEN MARKA='Kayıtsız Araç' AND MODEL LIKE 'MAN%'                                            THEN 'MAN'
            WHEN MARKA='Kayıtsız Araç'                                                                  THEN 'Diğer'
            ELSE MARKA
        END AS marka,
        COUNT(*) AS sayi
    FROM ariza
    WHERE GARAJADI IN ({ph_g})
    GROUP BY GARAJADI, marka
''', top_garaj_isimleri)

from collections import defaultdict
garaj_marka = defaultdict(dict)
for g, m, s in cur.fetchall():
    if m:
        garaj_marka[g][m] = s

# 3) Birleştir
garaj_bazli = []
for g, arac, ariza, ciddi in garaj_kpi:
    ciddi_per = round(ciddi / arac, 1) if arac else 0
    ariza_per = round(ariza / arac, 1) if arac else 0
    ciddi_pct = round(ciddi / ariza * 100, 1) if ariza else 0
    # Marka dağılımı: top 5
    mk_top = sorted(garaj_marka[g].items(), key=lambda x: -x[1])[:5]
    garaj_bazli.append({
        'garaj':           g,
        'sayi':            ariza,          # geriye uyumluluk için
        'ciddi':           ciddi,
        'ciddi_pct':       ciddi_pct,
        'arac':            arac,
        'ariza_per_arac':  ariza_per,
        'ciddi_per_arac':  ciddi_per,
        'markalar':        [{'marka': m, 'sayi': s} for m, s in mk_top],
    })

# Sort: ciddi_per_arac DESC (asıl risk metriği)
garaj_bazli.sort(key=lambda x: -x['ciddi_per_arac'])

print(f'GARAJ_BAZLI top 12 (ciddi/arac DESC):')
print(f'{"Garaj":30s} {"Araç":>5s} {"Arıza":>7s} {"Ciddi/Araç":>11s} {"Ciddi%":>7s}  Top 3 Marka')
for g in garaj_bazli[:12]:
    mk_str = ' | '.join(f'{m["marka"]}({m["sayi"]:,})' for m in g['markalar'][:3])
    print(f'  {g["garaj"]:30s} {g["arac"]:>5,} {g["sayi"]:>7,} {g["ciddi_per_arac"]:>11.1f} {g["ciddi_pct"]:>6.1f}%  {mk_str}')

GARAJ_BAZLI top 12 (ciddi/arac DESC):
Garaj                           Araç   Arıza  Ciddi/Araç  Ciddi%  Top 3 Marka
  HASANPASAGARAJI                  345  10,390        14.9   49.4%  MERCEDES(7,137) | OTOKAR(3,189)
  EDIRNEKAPIGARAJI                 382   9,328        12.5   51.2%  MERCEDES(7,064) | AKIA(2,264)
  IKITELLIISLETTIRMEGARAJI2        399  10,135         8.8   34.6%  KARSAN(5,521) | TEMSA(2,797) | BMC(1,249)
  KURTKOYGARAJI                    515   9,370         6.0   33.0%  OTOKAR(5,045) | KARSAN(3,375) | TEMSA(950)
  SULTANGAZİ GARAJI                502   8,398         5.9   35.3%  OTOKAR(5,872) | MERCEDES(2,259)
  Yunus (Park)                     249   3,430         3.7   26.9%  OTOKAR(3,359) | GÜLERYÜZ(71)
  SARIGAZIGARAJI                   249   2,578         2.9   27.8%  OTOKAR(2,285) | GÜLERYÜZ(293)
  ANADOLUGARAJI                    627   6,290         2.7   27.3%  MERCEDES(5,303) | OTOKAR(987)
  SAHINKAYAGARAJI                  215   1,708         2.4   30.6%  MERC

## 6) Marka Bazlı Arıza Sayısı

In [16]:
# MARKA bazlı arıza — MARKA='Kayıtsız Araç' kayıtlarında MODEL alanı %99.7 dolu.
# MODEL'in ilk kelimesinden gerçek MARKA tespit edilebilir:
#   'BMC PROCITY 285'      → BMC
#   'OTOKAR KENT'          → OTOKAR
#   'GÜLERYÜZ COBRA GD'    → GÜLERYÜZ
#   'MERCEDES CONECTO'     → MERCEDES
#   'AKIA ULTRA LF12'      → AKIA
# Bu CASE WHEN ile %99+ kayıt geri kazanılır (eskiden Top 12'de "Kayıtsız Araç" 9,579 ile 4. sıradaydı).

cur.execute("""
    SELECT
        CASE
            WHEN MARKA = 'Kayıtsız Araç' AND (MODEL LIKE 'BMC%' OR MODEL = 'BMC')                                    THEN 'BMC'
            WHEN MARKA = 'Kayıtsız Araç' AND (MODEL LIKE 'OTOKAR%' OR MODEL = 'OTOKAR')                              THEN 'OTOKAR'
            WHEN MARKA = 'Kayıtsız Araç' AND (MODEL LIKE 'MERCEDES%' OR MODEL = 'MERCEDES' OR MODEL LIKE 'CİTİPORT%') THEN 'MERCEDES'
            WHEN MARKA = 'Kayıtsız Araç' AND (MODEL LIKE 'TEMSA%' OR MODEL = 'TEMSA')                                THEN 'TEMSA'
            WHEN MARKA = 'Kayıtsız Araç' AND (MODEL LIKE 'GÜLERYÜZ%' OR MODEL LIKE 'COBRA%')                         THEN 'GÜLERYÜZ'
            WHEN MARKA = 'Kayıtsız Araç' AND (MODEL LIKE 'KARSAN%' OR MODEL = 'KARSAN')                              THEN 'KARSAN'
            WHEN MARKA = 'Kayıtsız Araç' AND (MODEL LIKE 'ANADOLU%' OR MODEL LIKE 'ISUZU%')                          THEN 'ISUZU'
            WHEN MARKA = 'Kayıtsız Araç' AND (MODEL LIKE 'AKIA%' OR MODEL = 'AKIA')                                  THEN 'AKIA'
            WHEN MARKA = 'Kayıtsız Araç' AND MODEL LIKE 'TEZELLER%'                                                  THEN 'TEZELLER'
            WHEN MARKA = 'Kayıtsız Araç' AND MODEL LIKE 'MAN%'                                                       THEN 'MAN'
            WHEN MARKA = 'Kayıtsız Araç' AND MODEL LIKE 'HABAS%'                                                     THEN 'HABAS'
            WHEN MARKA = 'Kayıtsız Araç'                                                                              THEN 'Bilinmiyor'  -- imputasyon yapilamayan (~29 kayit)
            ELSE MARKA
        END AS marka_temiz,
        COUNT(*) AS sayi
    FROM ariza
    WHERE MARKA IS NOT NULL AND MARKA != ''
    GROUP BY marka_temiz
    HAVING marka_temiz != 'Bilinmiyor'  -- Bilinmiyor goster me
    ORDER BY sayi DESC
    LIMIT 12
""")
marka_bazli = [{'marka': r[0], 'sayi': r[1]} for r in cur.fetchall()]
print(f'MARKA bazli ({len(marka_bazli)} marka):')
for m in marka_bazli:
    print(f'  {m["marka"]:20s} {m["sayi"]:>7,}')
marka_bazli[:5]

MARKA bazli (12 marka):
  MERCEDES              25,763
  OTOKAR                21,280
  KARSAN                11,792
  BMC                    8,791
  TEMSA                  4,232
  AKIA                   3,694
  GÜLERYÜZ               1,699
  ISUZU                    523
  TEZELLER                 191
  CLEANVAC                 149
  GREEN CAR                 60
  SGMS                      51


[{'marka': 'MERCEDES', 'sayi': 25763},
 {'marka': 'OTOKAR', 'sayi': 21280},
 {'marka': 'KARSAN', 'sayi': 11792},
 {'marka': 'BMC', 'sayi': 8791},
 {'marka': 'TEMSA', 'sayi': 4232}]

## 7) Operatör Dağılımı (İETT vs ÖHO vs KOOP)

In [17]:
cur.execute('SELECT USTOPERATORADI,COUNT(*) FROM ariza WHERE USTOPERATORADI IS NOT NULL GROUP BY USTOPERATORADI ORDER BY 2 DESC')
operator_dagilim = [{'operator': r[0], 'sayi': r[1]} for r in cur.fetchall()]
operator_dagilim

[{'operator': 'İETT', 'sayi': 68724},
 {'operator': 'ÖHO', 'sayi': 5799},
 {'operator': 'KOOP', 'sayi': 3751},
 {'operator': 'Kayıtsız Araç', 'sayi': 26}]

## 8) JSON Kaydet

In [18]:
out = {
    'kpi': kpi,
    'tip_bazli': tip_bazli,
    'saat_bazli': saat_bazli,
    'aylik_trend': aylik,
    'garaj_bazli': garaj_bazli,
    'marka_bazli': marka_bazli,
    'operator_dagilim': operator_dagilim,
    'meta': {
        'kaynak': 'panel_data/iett_data.db -> ariza tablosu',
        'donem': '2025 Oca-Haz',
        'toplam_kayit': toplam,
        'ciddi_tanim': 'SONUCTIPI IN (Yol Ustasi-Garaj, Cekilerek-Garaj, Telefonla-Garaj, Kayitci-Garaj, Oto Degisimi)',
        'mark_imputation': 'MARKA=Kayitsiz Arac kayitlari icin MODEL ilk kelimesi MARKA olarak atandi (%99+ basari)',
        'garaj_metrik': 'ciddi_per_arac (araç başına ciddi arıza) — adil oran',
    },
}

with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=1)

# Panel kopyası (Flask buradan okur)
import shutil
PANEL_PATH = Path(r'c:\Users\asus\Desktop\iett_panel\panel_data\ariza_ozet.json')
if PANEL_PATH.parent.exists():
    shutil.copy2(OUT_PATH, PANEL_PATH)
    print(f'Datathon: {OUT_PATH} ({OUT_PATH.stat().st_size:,} bytes)')
    print(f'Panel:    {PANEL_PATH} ({PANEL_PATH.stat().st_size:,} bytes)')
else:
    print(f'OK -> {OUT_PATH} yazıldı ({toplam:,} kayıt)')
    print(f'UYARI: Panel dizini bulunamadi, manuel kopya gerekli')

con.close()

Datathon: c:\Users\asus\Desktop\Datathon\panel_data\ariza_ozet.json (8,799 bytes)
Panel:    c:\Users\asus\Desktop\iett_panel\panel_data\ariza_ozet.json (8,799 bytes)
